# Reconstruccion de los FASTA de lncRNA (`outLncRNA_new.fa` y `lncRNA_2d_ss.fa`)

Este notebook **regenera desde cero** los dos archivos FASTA que necesita
`gen_lnc_characterization.ipynb` en sus celdas 2 y 3.

`gen_lnc_characterization.ipynb` lee:
- `outLncRNA_new.fa` -> secuencia de cada lncRNA  (funcion `read_fa`)
- `lncRNA_2d_ss.fa`  -> estructura 2D de cada lncRNA (funcion `read_fa2`)

y los vuelve a unir en `data/sequences/seq+ss_join_*.txt`. Como esos archivos
`seq+ss_join_*.txt` YA existen (linea 1 = secuencia, linea 2 = estructura 2D),
podemos invertir el proceso y reconstruir los dos `.fa` a partir de ellos.

## Archivos de ENTRADA
- `data/sequences/seq+ss_join_<lncRNA>.txt`  -> linea 1 = secuencia, linea 2 = estructura 2D
- `data/txt_interac/*.txt`  (opcional, para filtrar a los lncRNA de los pares)

## Archivos de SALIDA
- `data/fasta/fasta_seq/outLncRNA_new.fa`  (secuencias)
- `data/fasta/fasta_2d/lncRNA_2d_ss.fa`    (estructuras 2D)

Los dos archivos quedan con las MISMAS claves (nombres de lncRNA), que es lo que
necesita `gen_lnc_characterization.ipynb` para emparejar secuencia y estructura.

> **Sobre el rendimiento:** `gen_lnc_characterization.ipynb` corre el programa
> `RNAeval` una vez por cada lncRNA. Procesar los 121.249 lncRNA tarda horas.
> Por eso este notebook permite limitar cuantos se reconstruyen (`MAX_LNC`):
> si reconstruyes solo N, `gen_lnc_characterization` solo procesara esos N.


In [1]:
import os

# ====================== CONFIGURACION ======================
# Rutas relativas a la carpeta src/ (igual que el resto de notebooks del proyecto)
SEQ_LNC_DIR = "./../data/sequences"   # archivos seq+ss_join_*.txt de lncRNA

OUT_SEQ_FA = "./../data/fasta/fasta_seq/outLncRNA_new.fa"  # secuencias
OUT_SS_FA  = "./../data/fasta/fasta_2d/lncRNA_2d_ss.fa"    # estructuras 2D

PAIRS_FILES = [
    "./../data/txt_interac/mirnas_lncrnas_validated_positive_pairs.txt",
    "./../data/txt_interac/negative_pairs.txt",
]

# ONLY_PAIRS:
#   True  -> solo los lncRNA que aparecen en los pares de interaccion (3.165)
#   False -> todos los lncRNA de la carpeta de secuencias (121.249)
ONLY_PAIRS = True

# MAX_LNC: limite de lncRNA a reconstruir.
#   Un numero (p. ej. 50) -> reconstruye solo esa cantidad (util para una
#                            corrida rapida de gen_lnc_characterization).
#   None                  -> reconstruye todos los disponibles.
MAX_LNC = 50
# ============================================================

In [2]:
PREFIX = "seq+ss_join_"


def read_seq_ss(path):
    """Lee un archivo seq+ss_join_*.txt: linea 1 = secuencia, linea 2 = estructura 2D."""
    with open(path) as f:
        lines = f.read().splitlines()
    seq = lines[0].strip() if len(lines) > 0 else ""
    ss  = lines[1].strip() if len(lines) > 1 else ""
    return seq, ss


def reconstruct(seq_dir, out_seq_fa, out_ss_fa, keep=None, limit=None):
    """Reconstruye los dos FASTA (secuencia y estructura 2D) desde seq_dir."""
    os.makedirs(os.path.dirname(out_seq_fa), exist_ok=True)
    os.makedirs(os.path.dirname(out_ss_fa), exist_ok=True)
    names, skipped = [], 0
    with open(out_seq_fa, "w") as fseq, open(out_ss_fa, "w") as fss:
        for fname in sorted(os.listdir(seq_dir)):
            if not fname.startswith(PREFIX):
                continue
            name = os.path.splitext(fname[len(PREFIX):])[0]
            if keep is not None and name not in keep:
                continue
            seq, ss = read_seq_ss(os.path.join(seq_dir, fname))
            if not seq or not ss:        # archivo incompleto: se omite
                skipped += 1
                continue
            fseq.write(f">{name}\n{seq}\n")
            fss.write(f">{name}\n{ss}\n")
            names.append(name)
            if limit is not None and len(names) >= limit:
                break
    print(f"  -> {out_seq_fa}")
    print(f"  -> {out_ss_fa}")
    print(f"     {len(names)} lncRNA reconstruidos, {skipped} omitidos (incompletos)")
    return names

In [3]:
# Conjunto de lncRNA de los pares de interaccion (si ONLY_PAIRS = True)
keep = None
if ONLY_PAIRS:
    keep = set()
    for pf in PAIRS_FILES:
        if os.path.exists(pf):
            for line in open(pf):
                parts = line.strip().split(",")
                if parts:
                    keep.add(parts[0])
    print(f"lncRNA en pares de interaccion: {len(keep)}")
else:
    print("ONLY_PAIRS = False -> se reconstruyen todos los lncRNA disponibles")

lncRNA en pares de interaccion: 3165


In [4]:
# --- Reconstruir los dos FASTA ---
print("Reconstruyendo outLncRNA_new.fa y lncRNA_2d_ss.fa ...")
names = reconstruct(SEQ_LNC_DIR, OUT_SEQ_FA, OUT_SS_FA, keep=keep, limit=MAX_LNC)

Reconstruyendo outLncRNA_new.fa y lncRNA_2d_ss.fa ...
  -> ./../data/fasta/fasta_seq/outLncRNA_new.fa
  -> ./../data/fasta/fasta_2d/lncRNA_2d_ss.fa
     50 lncRNA reconstruidos, 0 omitidos (incompletos)


In [5]:
# --- Verificacion: los dos .fa deben tener las mismas claves ---
from Bio import SeqIO

def read_fa(path):
    return {str(x.id): str(x.seq) for x in SeqIO.parse(path, format="fasta")}

seq_fa = read_fa(OUT_SEQ_FA)
ss_fa  = read_fa(OUT_SS_FA)

print("=== VERIFICACION ===")
print(f"outLncRNA_new.fa : {len(seq_fa)} secuencias")
print(f"lncRNA_2d_ss.fa  : {len(ss_fa)} estructuras")
print(f"claves coincidentes: {set(seq_fa) == set(ss_fa)}")

ej = names[0]
print(f"\nEjemplo  {ej}")
print(f"  secuencia (len {len(seq_fa[ej])}): {seq_fa[ej][:60]}...")
print(f"  2D        (len {len(ss_fa[ej])}): {ss_fa[ej][:60]}...")

if set(seq_fa) == set(ss_fa) and len(seq_fa) > 0:
    print("\nOK: los .fa estan listos. Ya puedes correr gen_lnc_characterization.ipynb")
    print(f"    (procesara {len(seq_fa)} lncRNA con RNAeval).")
else:
    print("\nATENCION: las claves no coinciden o no se genero nada; revisa la configuracion.")

=== VERIFICACION ===
outLncRNA_new.fa : 50 secuencias
lncRNA_2d_ss.fa  : 50 estructuras
claves coincidentes: True

Ejemplo  NONHSAT000043.2
  secuencia (len 2513): CTGATCCATATGAATTCCTCTTATTAAGAAAAATAAAGCATCCAGGATTCAATGAAGAAC...
  2D        (len 2513): .........................................((((((.....((((((.....

OK: los .fa estan listos. Ya puedes correr gen_lnc_characterization.ipynb
    (procesara 50 lncRNA con RNAeval).
